# Despesas CEAP - Câmara/Brasil

O ZIP anual da CEAP é baixado uma vez e distribuído para as 27 UFs.

Esta versão também guarda, na raiz do Volume, os registros que não puderam ser associados a uma das 27 siglas em `_despesas_nao_distribuidas_YYYY.csv`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil
import tempfile
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "camara"
ID_LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
# diretório temporário exclusivo desta execução
# evita conflito de permissão em retry/serverless
TMP = Path(tempfile.mkdtemp(prefix="pi_ii_bronze_camara_brasil_"))

# Se False, anos que já estiverem completos são reaproveitados.
FORCAR_RECOLETA = False

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "*/*",
    "User-Agent": "PI-II-Univesp-Bronze-Camara-Brasil/1.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def pasta_uf(uf):
    return ROOT / uf.lower()

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def nome_normalizado(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

def encontrar_coluna(df, candidatos, contem_todos=None):
    mapa = {nome_normalizado(c): c for c in df.columns}

    for nome in candidatos:
        chave = nome_normalizado(nome)
        if chave in mapa:
            return mapa[chave]

    if contem_todos:
        termos = [nome_normalizado(x) for x in contem_todos]
        for c in df.columns:
            nc = nome_normalizado(c)
            if all(t in nc for t in termos):
                return c

    return None

def baixar(url, destino, timeout=(30, 900)):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    parcial = destino.with_suffix(destino.suffix + ".part")
    if parcial.exists():
        parcial.unlink()

    print("Baixando:", url)

    with session.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length") or 0)
        recebido = 0
        ultimo_print = time.time()

        with parcial.open("wb") as f:
            for bloco in r.iter_content(chunk_size=1024 * 1024):
                if not bloco:
                    continue

                f.write(bloco)
                recebido += len(bloco)

                if time.time() - ultimo_print >= 5:
                    if total:
                        print(
                            f"  {recebido / 1024**2:.1f} MB / "
                            f"{total / 1024**2:.1f} MB"
                        )
                    else:
                        print(f"  {recebido / 1024**2:.1f} MB")
                    ultimo_print = time.time()

    parcial.replace(destino)
    print(f"Concluído: {destino.name} ({destino.stat().st_size / 1024**2:.1f} MB)")
    return destino

def ler_csv_chunks(path, chunksize=100_000):
    return pd.read_csv(
        path,
        sep=";",
        encoding="utf-8",
        dtype=str,
        chunksize=chunksize,
        keep_default_na=False,
        na_filter=False,
    )

def append_local(df, path, cabecalho):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
        mode="a",
        header=cabecalho,
    )

def copiar_para_volume(origem, destino):
    origem = Path(origem)
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    with origem.open("rb") as src, destino.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024 * 8)

def criar_csv_vazio(path, colunas):
    pd.DataFrame(columns=colunas).to_csv(
        path,
        sep=";",
        index=False,
        encoding="utf-8",
    )

def limpar_tmp(prefixo):
    pasta = TMP / prefixo
    pasta.mkdir(parents=True, exist_ok=True)
    return pasta

def carregar_deputados_brasil():
    partes = []

    for uf in UFS:
        path = pasta_uf(uf) / "deputados.csv"
        if not path.exists():
            raise FileNotFoundError(
                f"{path} não existe. Execute primeiro o notebook 00."
            )

        df = pd.read_csv(
            path,
            sep=";",
            dtype=str,
            keep_default_na=False,
        )
        partes.append(df)

    deputados = pd.concat(partes, ignore_index=True).drop_duplicates()

    id_col = encontrar_coluna(
        deputados,
        ["id", "idDeputado", "deputado_id"],
    )
    uf_col = encontrar_coluna(
        deputados,
        ["siglaUf", "uf", "deputado_siglaUf"],
    )

    if not id_col or not uf_col:
        raise RuntimeError(
            "Não encontrei as colunas de ID/UF em deputados.csv."
        )

    mapa = (
        deputados[[id_col, uf_col]]
        .assign(
            **{
                id_col: deputados[id_col].astype(str).str.strip(),
                uf_col: deputados[uf_col].astype(str).str.strip().str.upper(),
            }
        )
        .drop_duplicates(subset=[id_col])
        .set_index(id_col)[uf_col]
        .to_dict()
    )

    return deputados, mapa

def ano_completo(arquivos):
    return all(Path(x).exists() for x in arquivos)

def salvar_manifesto(nome, payload):
    payload = dict(payload)
    payload["gerado_em_utc"] = utc_now()

    path = ROOT / f"_manifest_{nome}_brasil.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Manifesto:", path)

print("Bronze Câmara:", ROOT)
print("UFs:", len(UFS))
print("Anos:", ANOS)


def marker_v3(nome, ano):
    return ROOT / f"_{nome}_v3_{ano}.ok"


In [ ]:
import zipfile

def preparar_arquivos_tmp(pasta_tmp, ano):
    return {
        uf: pasta_tmp / f"despesas_{uf.lower()}_{ano}.csv"
        for uf in UFS
    }


In [ ]:
resumo = []

for ano in ANOS:
    print("\n" + "=" * 70)
    print("ANO", ano)

    marker = marker_v3("despesas", ano)

    esperados = [
        pasta_uf(uf) / f"despesas_{ano}.csv"
        for uf in UFS
    ]
    audit_destino = ROOT / f"_despesas_nao_distribuidas_{ano}.csv"

    if (
        not FORCAR_RECOLETA
        and marker.exists()
        and ano_completo(esperados)
        and audit_destino.exists()
    ):
        print("Ano já revisado pela v3; pulando.")
        resumo.append({"ano": ano, "status": "ja_existia_v3"})
        continue

    pasta_tmp = limpar_tmp(f"despesas_{ano}")

    url = f"https://www.camara.leg.br/cotas/Ano-{ano}.csv.zip"
    zip_local = pasta_tmp / f"Ano-{ano}.csv.zip"
    baixar(url, zip_local)

    extracao = pasta_tmp / "extraido"
    extracao.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_local, "r") as z:
        csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not csvs:
            raise RuntimeError(
                f"Nenhum CSV encontrado no ZIP de {ano}."
            )

        nome_csv = csvs[0]
        z.extract(nome_csv, extracao)

    csv_local = extracao / nome_csv
    tmp_por_uf = preparar_arquivos_tmp(pasta_tmp, ano)

    tmp_nao_distribuidas = pasta_tmp / f"despesas_nao_distribuidas_{ano}.csv"
    header_nao_distribuidas = True

    header = {uf: True for uf in UFS}
    contagens = {uf: 0 for uf in UFS}
    colunas = None
    total = 0
    total_nao_distribuido = 0
    valores_uf_nao_distribuidos = {}

    for chunk in ler_csv_chunks(csv_local):
        if colunas is None:
            colunas = list(chunk.columns)

        total += len(chunk)

        uf_col = encontrar_coluna(
            chunk,
            ["sgUF", "siglaUf", "uf"],
        )
        if not uf_col:
            raise RuntimeError(
                "Não encontrei a coluna de UF na CEAP. "
                f"Colunas: {list(chunk.columns)}"
            )

        serie_uf = (
            chunk[uf_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        mask_distribuida = serie_uf.isin(UFS)

        recorte = chunk.loc[mask_distribuida].copy()
        recorte["_uf_recorte"] = serie_uf.loc[recorte.index].values

        for uf, grupo in recorte.groupby("_uf_recorte"):
            grupo = grupo.drop(columns=["_uf_recorte"])

            append_local(
                grupo,
                tmp_por_uf[uf],
                cabecalho=header[uf],
            )
            header[uf] = False
            contagens[uf] += len(grupo)

        nao_distribuidas = chunk.loc[~mask_distribuida].copy()
        if not nao_distribuidas.empty:
            nao_distribuidas["_uf_normalizada"] = (
                serie_uf.loc[nao_distribuidas.index].values
            )

            append_local(
                nao_distribuidas,
                tmp_nao_distribuidas,
                cabecalho=header_nao_distribuidas,
            )
            header_nao_distribuidas = False
            total_nao_distribuido += len(nao_distribuidas)

            for valor, qtd in (
                nao_distribuidas["_uf_normalizada"]
                .value_counts(dropna=False)
                .to_dict()
                .items()
            ):
                chave = str(valor)
                valores_uf_nao_distribuidos[chave] = (
                    valores_uf_nao_distribuidos.get(chave, 0) + int(qtd)
                )

    for uf in UFS:
        destino = pasta_uf(uf) / f"despesas_{ano}.csv"
        local = tmp_por_uf[uf]

        if local.exists():
            copiar_para_volume(local, destino)
        else:
            criar_csv_vazio(destino, colunas or [])

        print(f"{uf}: {contagens[uf]:,}")

    if tmp_nao_distribuidas.exists():
        copiar_para_volume(tmp_nao_distribuidas, audit_destino)
    else:
        criar_csv_vazio(
            audit_destino,
            (colunas or []) + ["_uf_normalizada"],
        )

    distribuido = sum(contagens.values())

    print("Total nacional:", total)
    print("Distribuído nas UFs:", distribuido)
    print("Não distribuído:", total_nao_distribuido)
    print("Valores de UF fora do recorte:", valores_uf_nao_distribuidos)

    marker.write_text(utc_now(), encoding="utf-8")

    resumo.append({
        "ano": ano,
        "status": "ok",
        "registros_brasil": total,
        "registros_distribuidos": distribuido,
        "registros_nao_distribuidos": total_nao_distribuido,
        "percentual_nao_distribuido": (
            round((total_nao_distribuido / total) * 100, 4)
            if total else 0
        ),
        "valores_uf_nao_distribuidos": valores_uf_nao_distribuidos,
        "por_uf": contagens,
    })

salvar_manifesto("despesas", {"resumo": resumo})

tabela = pd.DataFrame([
    {
        "ano": x["ano"],
        "status": x["status"],
        "registros_brasil": x.get("registros_brasil"),
        "registros_distribuidos": x.get("registros_distribuidos"),
        "registros_nao_distribuidos": x.get("registros_nao_distribuidos"),
        "percentual_nao_distribuido": x.get("percentual_nao_distribuido"),
    }
    for x in resumo
])

if "display" in globals():
    display(tabela)
else:
    print(tabela.to_string(index=False))
